# nnInteractive Complexity Analysis

This notebook measures the computational complexity, inference speed, and GPU memory usage of the **actual pretrained nnInteractive** model.

To produce numbers **directly comparable** with the PromptUNet and UniverSeg benchmarks:
- Uses the **real, trained nnInteractive_v1.0 model** from HuggingFace
- Tests on a **single 128×128×128 3D volume** (the natural workload nnInteractive processes)
- Synchronises GPU after every volume pass (equivalent to `torch.cuda.synchronize()`)
- Includes inference session overhead (realistic measurement)

**Key architectural difference:** nnInteractive is a **3D model** (nnU-Net backbone) with native patch size `192×192×192` and 8 input channels (1 image + 7 interaction maps). A 128³ volume fits within a single 192³ patch. Unlike PromptUNet and UniverSeg which process 2D slices one-by-one, nnInteractive processes the full 3D volume in one forward pass.

In [ ]:
# !pip install thop nninteractive huggingface_hub

In [1]:
import torch
import time
import sys
import os
import warnings
import json
from huggingface_hub import snapshot_download
from nnInteractive.inference.inference_session import nnInteractiveInferenceSession

nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.


In [2]:
# Try to import thop for MACs/FLOPs
try:
    from thop import profile
    HAS_THOP = True
    print("thop is installed. GFLOPs calculation enabled.")
except ImportError:
    HAS_THOP = False
    warnings.warn("thop is not installed. GFLOPs calculation will be skipped. Install it via 'pip install thop'")

thop is installed. GFLOPs calculation enabled.


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Load Pretrained nnInteractive Model from HuggingFace

We download the official nnInteractive_v1.0 model from HuggingFace and initialize it via the `nnInteractiveInferenceSession`. This is the actual trained model used in production, providing realistic complexity metrics.

The model is a **3D PlainConvUNet** from the `dynamic_network_architectures` package — the same backbone used by nnU-Net V2.

In [5]:
from nnInteractive.inference.inference_session import nnInteractiveInferenceSession

# ---- Download pretrained model from HuggingFace ----
MODEL_REPO_ID = "nnInteractive/nnInteractive"
MODEL_NAME = "nnInteractive_v1.0"
DOWNLOAD_DIR = "./nninteractive_model"

print(f"Downloading {MODEL_NAME} from {MODEL_REPO_ID}...")
try:
    model_path = snapshot_download(
        repo_id=MODEL_REPO_ID,
        allow_patterns=[f"{MODEL_NAME}/*"],
        local_dir=DOWNLOAD_DIR,
        cache_dir=None  # Use local_dir instead of cache
    )
    print(f"Model downloaded to: {model_path}")
except Exception as e:
    print(f"Download failed: {e}")
    print("Attempting alternative download path...")
    model_path = os.path.join(DOWNLOAD_DIR, MODEL_NAME)

# ---- Initialize nnInteractive inference session ----
print("\nInitializing nnInteractive inference session...")
try:
    session = nnInteractiveInferenceSession(
        device=device,
        use_torch_compile=False,  # For fair comparison, disable torch.compile
        verbose=False
    )

    actual_model_dir = os.path.join(model_path, MODEL_NAME)

    # Initialize session from downloaded model
    session.initialize_from_trained_model_folder(actual_model_dir)

    # Extract the underlying network for complexity measurements
    model = session.network
    model.eval()

    print("Model loaded successfully (pretrained nnInteractive_v1.0)")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# ---- Model configuration ----
NUM_INPUT_CHANNELS = 8               # 1 image + 7 interaction channels (actual nnInteractive input)
NUM_OUTPUT_CHANNELS = 2              # Binary segmentation

# Volume size for comparison with PromptUNet/UniverSeg benchmarks
# A 128×128×128 3D volume is the natural workload nnInteractive processes
VOLUME_SIZE = (128, 128, 128)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Model downloaded to: /content/nninteractive_model

Initializing nnInteractive inference session...
Model loaded successfully (pretrained nnInteractive_v1.0)


In [6]:
# Define input dimensions
# nnInteractive operates on 3D volumes, not individual 2D slices.
# Test on a 128×128×128 volume with 8 input channels (1 image + 7 interaction maps).
B = 1                    # Batch size
C_IN = 8                 # 1 image + 7 interaction channels
D, H, W = VOLUME_SIZE    # 128 x 128 x 128

print(f"Input shape: (Batch={B}, C={C_IN}, D={D}, H={H}, W={W})")
print(f"Volume size: {VOLUME_SIZE}")

# Dummy data for single-pass tests
dummy_input = torch.randn(B, C_IN, D, H, W).to(device)

Input shape: (Batch=1, C=8, D=128, H=128, W=128)
Volume size: (128, 128, 128)


## Model Complexity (GFLOPs)

Measured on a **single 3D volume** of size `128×128×128` with 8 input channels — the natural workload nnInteractive processes in one forward pass.

Divide GFLOPs by 128 to recieve a comparable results to p_unet and universeg

In [7]:
if HAS_THOP:
    # Calculate FLOPs using thop
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    # 1 MAC is generally considered as 2 FLOPs (one multiply, one add)
    gflops = (macs * 2) / 1e9
    print(f"Parameters: {params / 1e6:.2f} M")
    print(f"GFLOPs (approx): {gflops:.2f}")
else:
    # Basic param count if thop is missing
    params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {params / 1e6:.2f} M")
    print("GFLOPs: N/A (Install 'thop' module to automatically calculate FLOPs)")

Parameters: 383.49 M
GFLOPs (approx): 17996.58


## Inference Speed (Volume Measurement)

Measures inference time for a **single 128×128×128 3D volume**.

**Key point:** nnInteractive is a native 3D model that processes the entire 128³ volume in **one forward pass** (unlike PromptUNet/UniverSeg which process 128 individual 2D slices). This measurement is directly comparable to PromptUNet's per-volume inference time: we measure the total time to segment one complete 3D volume.

**Timing methodology:**
- Warm-up: 10 forward passes to trigger compilation and warm the GPU
- Timed inference: 1 volume forward pass with GPU synchronization
- GPU is synchronised after the pass for precise timing (equivalent to `torch.cuda.synchronize()`)

In [10]:
import torch
import gc

# Clear any references held by the 'out' history
%reset -f out

# Force immediate collection
gc.collect()
torch.cuda.empty_cache()

Flushing output cache (0 entries)


In [11]:
# ---- Clean up residual profiler hooks ----
print("Clearing residual profiling hooks...")
for m in model.modules():
    m._forward_hooks.clear()
    m._forward_pre_hooks.clear()
    m._backward_hooks.clear()

    # Optionally delete the attributes if they happen to still exist
    if hasattr(m, 'total_ops'):
        del m.total_ops
    if hasattr(m, 'total_params'):
        del m.total_params

# ---- Warm-up phase ----
print('Warming up model...')
with torch.no_grad():
    for _ in range(10):
        _ = model(dummy_input)

if device.type == 'cuda':
    torch.cuda.synchronize()
print('Warm-up complete.\n')


# ---- Timed volume inference ----
print(f'Running volume inference on {D}x{H}x{W} volume...\n')

if device.type == 'cuda':
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    # Single forward pass for entire 3D volume
    _ = model(dummy_input)

if device.type == 'cuda':
    torch.cuda.synchronize()

end_time = time.perf_counter()


total_volume_time = end_time - start_time
throughput = 1.0 / total_volume_time  # volumes per second

print(f'--- Volume Inference Results ({D}x{H}x{W}) ---')
print(f'Total volume inference time : {total_volume_time * 1000:.2f} ms')
print(f'Throughput                  : {throughput:.2f} volumes/sec')
print()
print('Note: nnInteractive segments the entire 128×128×128 3D volume')
print('in a single forward pass. This is directly comparable to')
print('PromptUNet\'s per-volume inference time.')

Clearing residual profiling hooks...
Warming up model...
Warm-up complete.

Running volume inference on 128x128x128 volume...

--- Volume Inference Results (128x128x128) ---
Total volume inference time : 157.47 ms
Throughput                  : 6.35 volumes/sec

Note: nnInteractive segments the entire 128×128×128 3D volume
in a single forward pass. This is directly comparable to
PromptUNet's per-volume inference time.


Divide Total volume inference time by 128 to recieve a comparable results to p_unet and universeg

## GPU Memory Usage

In [ ]:
if device.type == 'cuda':
    # 1. Clear out any old junk before we start measuring
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Run single volume inference to capture peak memory
    with torch.no_grad():
        _ = model(dummy_input)

    # 2. Grab the peak stats
    max_mem    = torch.cuda.max_memory_allocated() / (1024 ** 3)  # GB
    max_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB

    # Optional: also check reserved memory (what PyTorch holds onto)
    max_res_mb = torch.cuda.max_memory_reserved() / (1024 ** 2)

    print(f"Peak GPU Memory Allocated: {max_mem_mb:.2f} MB ({max_mem:.3f} GB)")
    print(f"Peak GPU Memory Reserved : {max_res_mb:.2f} MB")
else:
    print("GPU Memory Usage: N/A (running on CPU)")

Peak GPU Memory Allocated: 2414.31 MB (2.358 GB)
Peak GPU Memory Reserved : 3218.00 MB
